## Imports

In [2]:
%matplotlib qt

import mindmap as mm
from pathlib import Path 
import mne

## Data

In [4]:
sub     = "sub-pat10"
dataset = "spike"

# configuration
config = mm.io.read_config("../configurations/spike.yml")
mm_dir = Path(config["MM_DIR"])

# files
#folder  = f"/home/admin/work/data/intra_recordings/derivatives/spike_trials/{sub}/eeg/"                                    # dataset: interelec
folder = f"/home/nino/Data/eeg_epi_spikeerp_data_1.0.0/derivatives/cartool-v3.80/{sub}/ses-preop/eeg/"                # dataset: spike

# data
raw = mm.io.read_eeg(folder, dataset)

## Raw

#### Visualize Raw

In [5]:
_ = raw.plot()

In [7]:
_ = raw.compute_psd().plot()

    Using multitaper spectrum estimation with 7 DPSS windows
Plotting power spectral density (dB=True).


#### Preprocess Raw

Interpolate bad channels

In [8]:
raw.info["bads"].append("7")
raw.interpolate_bads()

Setting channel interpolation method to {'eeg': 'spline'}.
Interpolating bad channels.
    Automatic origin fit: head of radius 88.7 mm
Computing interpolation matrix from 203 sensor positions
Interpolating 1 sensors


Condition,
Data kind,average
Timepoints,250 samples
Channels,204 channels
Number of averaged epochs,1
Time range (secs),0.0 – 0.996
Baseline (secs),off


## Epoched

## Evoked


In [8]:
if dataset == "spike":
    evoked = raw.copy()

elif dataset == "intra":
    raw = raw.set_eeg_reference(["VREF"])
    evoked = raw.average()
    evoked = mne.set_eeg_reference(evoked, "average")

_ = evoked.plot()

#### Preprocess Evoked

Baseline correction

In [ ]:
baseline = [0, 0.3]
evoked.apply_baseline(baseline)

_ = evoked.plot()

Notch filtering

In [ ]:
evoked.filter(
    l_freq = 63,
    h_freq = 37,
    picks  = "all",
    method = "fir",
)

_ = evoked.plot()

## Visualize Topomaps

In [9]:
# time points 
times = [0.4, 0.45, 0.5, 0.55, 0.6]

_ = evoked.plot_topomap(
    times = times,
    outlines = None,
    extrapolate = "local",
)


## Save processed EEG

In [10]:
# save source space
folder = mm_dir/sub/"eeg"

if folder.exists() == False:
        folder.mkdir(parents=True)

print(f"Save EEG for subject: {sub}")
evoked.save(folder/f"{sub}_clean-ave.fif", overwrite=True)

Save EEG for subject: sub-pat10
